# 04 · Price model and SHAP explanations

**Question:** what drives the price per m² of a Paris apartment, and which areas sell below what their characteristics would predict?

- Training and comparison happen in `src/train_model.py` (run it first). This notebook reads the results and explains the best model.
- **Target:** log(price per m²), converted back to €/m² for every metric.
- **Time-based split:** train on sales up to 12 months before the last sale, test on the last 12 months. A random split would let the model learn from sales that happened *after* the ones it predicts, which overstates accuracy.
- **Features:** surface, rooms, arrondissement, latitude/longitude, distance to metro/RER, distance to centre, has a cellar/parking, months since start.
- **Limitation:** DVF has no floor, condition, view, balcony or energy rating (DPE). These explain much of the price gap between two flats in the same street, so a large part of the error is irreducible with this data.

In [ ]:
import sys
from pathlib import Path

import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from xgboost import XGBRegressor

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from train_model import FEATURES, TARGET, load_dataset, time_split  # noqa: E402

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR.mkdir(parents=True, exist_ok=True)
BLUE = "#3b6ea5"
sns.set_theme(style="whitegrid")

con = duckdb.connect(str(PROJECT_ROOT / "data" / "paris_dvf.duckdb"), read_only=True)
df = load_dataset(con)
train, test, cutoff = time_split(df)
print(f"Train: {len(train):,} sales up to {cutoff:%Y-%m-%d} | Test: {len(test):,} sales after")

## 1. Model comparison (test set = last 12 months)

In [ ]:
metrics_df = pd.read_csv(PROJECT_ROOT / "reports" / "model_metrics.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].barh(metrics_df["model"], metrics_df["MAE_eur_m2"], color=BLUE)
axes[0].set_title("MAE (€ per m², lower is better)"); axes[0].invert_yaxis()
axes[1].barh(metrics_df["model"], metrics_df["R2"], color=BLUE)
axes[1].set_title("R² (higher is better)"); axes[1].invert_yaxis(); axes[1].set_yticklabels([])
fig.tight_layout(); fig.savefig(FIG_DIR / "model_comparison.png", dpi=150); plt.show()
metrics_df

**Insight:** _to be written from the real output above._

## 2. Predicted vs actual (XGBoost, test set)

In [ ]:
preds = con.execute("SELECT * FROM model_predictions").df()
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(preds["predicted_price_m2"], preds["actual_price_m2"], s=3, alpha=0.15, color=BLUE)
lims = [preds[["predicted_price_m2", "actual_price_m2"]].min().min(), preds[["predicted_price_m2", "actual_price_m2"]].max().max()]
ax.plot(lims, lims, color="black", linewidth=1)
ax.set_xlabel("Predicted € per m²"); ax.set_ylabel("Actual € per m²")
ax.set_title("XGBoost: predicted vs actual (last 12 months)")
fig.tight_layout(); fig.savefig(FIG_DIR / "predicted_vs_actual.png", dpi=150); plt.show()

err = (preds["predicted_price_m2"] / preds["actual_price_m2"] - 1).abs()
print(f"Sales predicted within 10%: {(err <= 0.10).mean():.1%} | within 20%: {(err <= 0.20).mean():.1%}")

**Insight:** _to be written from the real output above._

## 3. SHAP: what drives the prediction?

SHAP splits each prediction into a contribution per feature. Because the model predicts **log** price per m², a SHAP value of +0.10 means roughly **+10%** on the price per m², relative to the average prediction.
Computed on a random sample of 3,000 test sales.

In [ ]:
xgb = XGBRegressor()
xgb.load_model(PROJECT_ROOT / "models" / "xgb_price_m2.json")

X_sample = test[FEATURES].sample(3000, random_state=42) if len(test) > 3000 else test[FEATURES]
explainer = shap.TreeExplainer(xgb)
sv = explainer(X_sample)

shap.plots.beeswarm(sv, max_display=len(FEATURES), show=False)
plt.title("SHAP summary: impact on log(price per m²)")
plt.gcf().savefig(FIG_DIR / "shap_summary.png", dpi=150, bbox_inches="tight"); plt.show()

importance = (pd.DataFrame({"feature": FEATURES, "mean_abs_shap": np.abs(sv.values).mean(axis=0)})
              .sort_values("mean_abs_shap", ascending=False))
importance["approx_avg_effect_pct"] = (100 * importance["mean_abs_shap"]).round(1)
importance.to_csv(PROCESSED_DIR / "agg_shap_importance.csv", index=False)
importance

**Insight:** _to be written from the real output above._

### Dependence plots: surface and distance to metro

In [ ]:
for feature, fname in [("surface_reelle_bati", "shap_dependence_surface.png"),
                       ("dist_metro_m", "shap_dependence_metro.png")]:
    shap.plots.scatter(sv[:, feature], color=sv[:, "dist_centre_m"], show=False)
    plt.title(f"SHAP dependence: {feature}")
    plt.gcf().savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight"); plt.show()

**Insight:** _to be written from the real output above._

### Waterfall: one example apartment

A typical sale from the test set: the one whose predicted price is closest to the test-set median.

In [ ]:
pred_sample = xgb.predict(X_sample)
i = int(np.argmin(np.abs(pred_sample - np.median(pred_sample))))
example = X_sample.iloc[i]
print(f"Arrondissement {example['arrondissement']:.0f}, {example['surface_reelle_bati']:.0f} m², "
      f"{example['nombre_pieces_principales']:.0f} rooms, {example['dist_metro_m']:.0f} m from metro")
print(f"Predicted: {np.exp(pred_sample[i]):,.0f} €/m²  |  Actual: {test.loc[X_sample.index[i], TARGET]:,.0f} €/m²")

shap.plots.waterfall(sv[i], max_display=len(FEATURES), show=False)
plt.gcf().savefig(FIG_DIR / "shap_waterfall_example.png", dpi=150, bbox_inches="tight"); plt.show()

**Insight:** _to be written from the real output above._

## 4. Q8 · Which areas sell below what the model predicts?

For each arrondissement: the median of (actual / predicted − 1) on the test set. Negative means sales happened **below** what comparable flats (same size, location, distances) would predict: a possible sign of under-pricing or of features the model can't see (condition, floor).

In [ ]:
gap = pd.read_csv(PROCESSED_DIR / "agg_model_gap_by_arr.csv")
arr = gpd.read_file(PROJECT_ROOT / "data" / "external" / "arrondissements.geojson")
cols = {c.lower(): c for c in arr.columns}
arr["arrondissement"] = arr[cols["c_ar"]].astype(int) if "c_ar" in cols else arr[cols["c_arinsee"]].astype(int) - 75100
arr = arr.merge(gap, on="arrondissement").to_crs("EPSG:2154")

lim = arr["median_gap_pct"].abs().max()
fig, ax = plt.subplots(figsize=(8, 6))
arr.plot(column="median_gap_pct", cmap="RdBu", vmin=-lim, vmax=lim, edgecolor="white", linewidth=0.6,
         legend=True, legend_kwds={"label": "Actual vs predicted (%)", "shrink": 0.6}, ax=ax)
for _, r in arr.iterrows():
    p = r.geometry.representative_point()
    ax.annotate(f"{r['arrondissement']}", (p.x, p.y), ha="center", va="center", fontsize=9)
ax.set_axis_off(); ax.set_title("Median gap between actual and predicted price per m² (last 12 months)")
fig.tight_layout(); fig.savefig(FIG_DIR / "map_model_gap.png", dpi=150, bbox_inches="tight"); plt.show()
gap

**Insight:** _to be written from the real output above._

## Key takeaways

_To be written from the real outputs above._

In [ ]:
con.close()